In [1]:
import pandas as pd
import numpy as np
from udrud_framework import estimate_gamma, calculate_ud_rud_metrics

# Load your empirical dataset
df = pd.read_csv("2021_2025_disposable_income.csv")

# equivalize income and weight
df["weight"] = df["weight"] * df["size"]
df["income"] = df["income"] / np.sqrt(df["size"])

# dataframes for results
summarydf = pd.DataFrame(columns=["year", "No Households", "No Negative Incomes",
                                  "No Zero Incomes", "Minimum Income"])
estimatedf = pd.DataFrame(columns=["year", "k^*", "gamma^"])
indicesdf = pd.DataFrame(columns=["year", "Indices"])

# analyze
for year, group in df.groupby("year"):
    new_summary = {"year": year,
                   "No Households": group["income"].count(),
                   "No Negative Incomes": np.sum(group["income"] < 0),
                    "No Zero Incomes": np.sum(group["income"] == 0),
                    "Minimum Income": group["income"].min()}
    summarydf.loc[len(summarydf)] = new_summary

    y_raw = group['income'].values
    weights = group['weight'].values

# Step 1: Estimate the true baseline (Algorithms 1-3)
    gamma_hat, optimal_k, plateau_series = estimate_gamma(y_raw, weights, p=0.05)

    new_estimate = {"year": year,
                    "k^*": optimal_k,  # <-- Updated to match column name
                    "gamma^":  gamma_hat}
    estimatedf.loc[len(estimatedf)] = new_estimate


# Step 2: Calculate all comparative metrics
    indices = calculate_ud_rud_metrics(y_raw, weights, gamma_hat)
    new_indices = {"year": year,
                   "Indices": indices}
    indicesdf.loc[len(indicesdf)] = new_indices

# View the dictionary to build your LaTeX tables!

#    for metric, value in results.items():
#        print(f"{metric}: {value:.4f}")
#


In [2]:
print(summarydf)
print()

print(estimatedf)
print()

for row in range(indicesdf.shape[0]):
    print(indicesdf.loc[row, 'year'])
    for metric, value in indicesdf.loc[row, 'Indices'].items():
        print(metric, value)
    print()



   year  No Households  No Negative Incomes  No Zero Incomes  Minimum Income
0  2021          18187                   68                1   -21976.000000
1  2022          17954                   81                0    -5584.022251
2  2023          18094                   84                1   -16155.975737
3  2024          18314                   87                1   -12746.306838
4  2025          18664                   66                0    -5998.000000

   year   k^*        gamma^
0  2021   987 -38180.719595
1  2022  1031  -5853.680246
2  2023  1038 -21090.102585
3  2024  1085 -18489.754062
4  2025   217  -6102.060354

2021
Baseline_Gamma -38180.719594758004
Mean_Income_y 3352.6588467321453
Conventional_Gini 0.33543812822560465
Conventional_CV 0.7914468655874954
RUD_Mean_mu_z 12.38819108659772
RUD_Variance_V_z 0.626388141048271
RUD_Gini_G_z 0.027077248476454426
RUD_CV_z 0.06388720193731348
L1_c (CDF) 12.38819108659772
L1_q (QF) 12.38819108659772
L1_cq (CDQF) 24.77638217319544
L2_c